# 07 - Visualization and Reporting

This notebook creates publication-quality figures and a comprehensive summary report of all EMS optimization results.

**Runtime:** ~3 minutes  
**Data Required:** Simulation results, statistical analysis outputs

In [ ]:
import sys, os

IN_COLAB = 'google.colab' in sys.modules
DOWNLOAD_OUTPUTS = False  # Set True to download output files
SAVE_TO_DRIVE = False     # Set True to save outputs to Google Drive

if IN_COLAB:
    print("Running in Google Colab - installing dependencies...")
    !pip install -q simpy pulp pyyaml tqdm
    if not os.path.exists('ems-optimization'):
        !git clone --depth=1 https://github.com/cnsp/ems-optimization.git
    PROJECT_ROOT = '/content/ems-optimization'
else:
    print("Running locally")
    PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))

sys.path.insert(0, os.path.join(PROJECT_ROOT, 'src'))
DATA_DIR = os.path.join(PROJECT_ROOT, 'data')
RAW_DIR = os.path.join(DATA_DIR, 'raw')
PROCESSED_DIR = os.path.join(DATA_DIR, 'processed')
RESULTS_DIR = os.path.join(PROJECT_ROOT, 'results')
CONFIGS_DIR = os.path.join(PROJECT_ROOT, 'configs')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100

# Optional Google Drive save
if IN_COLAB and SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_DIR = '/content/drive/MyDrive/EMS_Optimization_Results'
    os.makedirs(DRIVE_DIR, exist_ok=True)
    print(f"Saving outputs to: {DRIVE_DIR}")

def save_output(fig_or_df, filename, subdir=''):
    """Helper to save outputs with optional download/drive save."""
    out_dir = os.path.join(RESULTS_DIR, subdir) if subdir else RESULTS_DIR
    os.makedirs(out_dir, exist_ok=True)
    filepath = os.path.join(out_dir, filename)
    if isinstance(fig_or_df, pd.DataFrame):
        fig_or_df.to_csv(filepath, index=True)
    elif hasattr(fig_or_df, 'savefig'):
        fig_or_df.savefig(filepath, bbox_inches='tight', dpi=150)
    if IN_COLAB and DOWNLOAD_OUTPUTS:
        from google.colab import files
        files.download(filepath)
    if IN_COLAB and SAVE_TO_DRIVE:
        import shutil
        drive_path = os.path.join(DRIVE_DIR, subdir)
        os.makedirs(drive_path, exist_ok=True)
        shutil.copy(filepath, os.path.join(drive_path, filename))

print("Setup complete. PROJECT_ROOT:", PROJECT_ROOT)

In [ ]:
plt.rcParams.update({
    'font.size': 12,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
})

## Load All Results

In [ ]:
# Simulation results
sim_path = os.path.join(RESULTS_DIR, 'simulation', 'simulation_results_all.csv')
if os.path.exists(sim_path):
    results_df = pd.read_csv(sim_path)
    print(f"Simulation results: {len(results_df)} rows")
else:
    print("WARNING: No simulation results found. Visualizations will be limited.")
    results_df = pd.DataFrame()

# Firehouses
firehouses = pd.read_csv(os.path.join(PROCESSED_DIR, 'firehouses_manhattan.csv'))
# Demand
precinct_demand = pd.read_csv(os.path.join(PROCESSED_DIR, 'demand_lambda_precinct.csv'))

print(f"Firehouses: {len(firehouses)}")
print(f"Precincts: {len(precinct_demand)}")

## Figure 1: Policy Comparison (Main Result)

In [ ]:
if not results_df.empty:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    policies = sorted(results_df['policy'].unique())
    colors = {'P0': '#4878CF', 'P1': '#E8A838', 'P2': '#6ACC65'}
    labels = {'P0': 'P0 (Spatial Baseline)', 'P1': 'P1 (Demand-Proportional)', 'P2': 'P2 (Optimized)'}

    n_reps = results_df.groupby(['policy', 'K']).size().max()

    # Response time
    for policy in policies:
        grp = results_df[results_df['policy'] == policy].groupby('K')['response_time_mean']
        means = grp.mean()
        stds = grp.std()
        axes[0].errorbar(means.index, means.values, yerr=1.96*stds/np.sqrt(n_reps),
                        fmt='-o', label=labels.get(policy, policy), color=colors.get(policy),
                        capsize=4, markersize=7, linewidth=2)
    axes[0].set_xlabel('Fleet Size (K)')
    axes[0].set_ylabel('Mean Response Time (min)')
    axes[0].set_title('(a) Response Time')
    axes[0].legend(fontsize=9)
    axes[0].grid(True, alpha=0.3)

    # Coverage
    for policy in policies:
        grp = results_df[results_df['policy'] == policy].groupby('K')['coverage_fraction']
        means = grp.mean()
        stds = grp.std()
        axes[1].errorbar(means.index, means.values * 100, yerr=1.96*stds*100/np.sqrt(n_reps),
                        fmt='-s', label=labels.get(policy, policy), color=colors.get(policy),
                        capsize=4, markersize=7, linewidth=2)
    axes[1].set_xlabel('Fleet Size (K)')
    axes[1].set_ylabel('Coverage (%)')
    axes[1].set_title('(b) Coverage (within 8 min)')
    axes[1].legend(fontsize=9)
    axes[1].grid(True, alpha=0.3)

    # Queue fraction
    for policy in policies:
        grp = results_df[results_df['policy'] == policy].groupby('K')['queue_fraction']
        means = grp.mean()
        axes[2].plot(means.index, means.values * 100, '-^', label=labels.get(policy, policy),
                    color=colors.get(policy), markersize=7, linewidth=2)
    axes[2].set_xlabel('Fleet Size (K)')
    axes[2].set_ylabel('Queue Fraction (%)')
    axes[2].set_title('(c) Queueing Rate')
    axes[2].legend(fontsize=9)
    axes[2].grid(True, alpha=0.3)

    plt.tight_layout()
    save_output(fig, 'fig1_policy_comparison.png', 'figures/publication')
    plt.show()

## Figure 2: Response Time Distributions

In [ ]:
if not results_df.empty:
    K_show = results_df['K'].mode().values[0]
    fig, axes = plt.subplots(1, len(policies), figsize=(5*len(policies), 5))
    if len(policies) == 1:
        axes = [axes]

    for idx, policy in enumerate(policies):
        data = results_df[(results_df['policy'] == policy) & (results_df['K'] == K_show)]['response_time_mean']
        axes[idx].hist(data, bins=15, density=True, color=colors.get(policy, 'gray'),
                      edgecolor='white', alpha=0.7)
        axes[idx].axvline(x=data.mean(), color='red', linestyle='--',
                         label=f'Mean: {data.mean():.2f} min')
        axes[idx].set_xlabel('Mean Response Time (min)')
        axes[idx].set_ylabel('Density')
        axes[idx].set_title(f'{labels.get(policy, policy)} (K={K_show})')
        axes[idx].legend()

    plt.tight_layout()
    save_output(fig, 'fig2_rt_distributions.png', 'figures/publication')
    plt.show()

## Figure 3: Firehouse Locations

In [ ]:
fig, ax = plt.subplots(figsize=(10, 14))

# Plot all firehouses
cbd_fh = firehouses[firehouses['in_cbd'] == True]
non_cbd_fh = firehouses[firehouses['in_cbd'] == False]

ax.scatter(non_cbd_fh['Longitude'], non_cbd_fh['Latitude'], c='steelblue',
          s=60, label=f'Non-CBD ({len(non_cbd_fh)})', zorder=5, edgecolors='white')
ax.scatter(cbd_fh['Longitude'], cbd_fh['Latitude'], c='red',
          s=80, marker='D', label=f'CBD ({len(cbd_fh)})', zorder=6, edgecolors='white')

ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_title('Manhattan FDNY Firehouse Locations')
ax.legend(fontsize=11)
ax.set_aspect('equal')
plt.tight_layout()
save_output(fig, 'fig3_firehouse_locations.png', 'figures/publication')
plt.show()

## Generate Summary Report

In [ ]:
report = []
report.append("# EMS Readiness Optimization - Results Summary")
report.append(f"\nGenerated: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M')}")
report.append("\n## Study Overview")
report.append(f"- Study area: Manhattan, New York City")
report.append(f"- Candidate staging locations: {len(firehouses)} FDNY firehouses")
report.append(f"- Demand zones: {len(precinct_demand)} precincts")
report.append(f"- Response threshold: 8 minutes (NFPA standard)")

if not results_df.empty:
    report.append("\n## Key Findings")
    policies = sorted(results_df['policy'].unique())
    K_vals = sorted(results_df['K'].unique())

    for K in K_vals:
        report.append(f"\n### Fleet Size K = {K}")
        for policy in policies:
            subset = results_df[(results_df['policy'] == policy) & (results_df['K'] == K)]
            if len(subset) > 0:
                rt = subset['response_time_mean'].mean()
                cov = subset['coverage_fraction'].mean() * 100
                report.append(f"- {policy}: Mean RT = {rt:.2f} min, Coverage = {cov:.1f}%")

    # Best policy
    best = results_df.groupby('policy')['response_time_mean'].mean()
    best_policy = best.idxmin()
    report.append(f"\n## Recommendation")
    report.append(f"- **{best_policy}** achieves the lowest mean response time across all fleet sizes")
    report.append(f"- Fleet sizes K=20-30 provide the best cost-effectiveness tradeoff")

report_text = "\n".join(report)
print(report_text)

# Save report
report_path = os.path.join(RESULTS_DIR, 'summary_report.md')
with open(report_path, 'w') as f:
    f.write(report_text)
print(f"\nReport saved to: {report_path}")

## Summary

All publication-quality figures and summary report have been generated.

Key outputs:
- `fig1_policy_comparison.png` - Main result figure
- `fig2_rt_distributions.png` - Response time distributions
- `fig3_firehouse_locations.png` - Spatial overview
- `summary_report.md` - Text summary of findings